# Parallel Tempering with Fulton Market

In this exercise, we will perform a parallel tempering calculation and some analysis. Parallel tempering is a type of replica exchange in which replicas differ only be temperature. Fulton Market is a package being developed my research group that implements several specific versions of replica exchange. This exercise was designed to be run on the SDSC Expanse high performance computing resource.

Assuming that you are reading this Notebook on Github, you will need to download it to your account on SDSC Expanse. To do that, log onto the Expanse User Portal, select *Shell*, and paste the following commands into the terminal:

```bash
mkdir -p ~/exercises
cd ~/exercises
curl -L -o 10-Parallel_Tempering.ipynb https://raw.githubusercontent.com/daveminh/Chem456-2026S/refs/heads/main/exercises/10-Parallel_Tempering.ipynb
```

We will need to set up a new environment before starting JupyterLab. You can do that by following the steps in Part 0.

The exercise will be graded based on submitting your answers to the questions after ```-->``` on Canvas.

# Part 0 - Conda environment setup on SDSC Expanse

First, we need to set up a new conda environment

## 1) Log onto the Expanse User Portal → *Shell*.

## 2) Start an interactive GPU job

We need to use a GPU job for installation so that the appropriate jax packages are installed.

```bash
srun --partition=gpu-debug --account=iit130 --nodes=1 --ntasks=1 --gpus 1 --cpus-per-task=4 --mem=8G --time=00:30:00 --pty bash
```

## 3) Create and register a conda environment

We’ll create a new environment named `fultonmarket`. Paste the following commands into the terminal:
```bash
source "$HOME/miniconda3/etc/profile.d/conda.sh"
mamba create -n fultonmarket -c conda-forge -y jupyter matplotlib mdanalysis mdtraj netcdf4 numba nglview==4.0.0 "numpy<2" openmm openmmtools pandas pdbfixer pip pymbar=3.1.1 pytables scikit-learn scipy seaborn python=3.11 && conda activate fultonmarket && pip install jax==0.4.25 jaxlib==0.4.25 jax-cuda11-pjrt==0.4.25 jax-cuda11-plugin==0.4.25 "numpy<2"
```
Installation should take 10-15 minutes. This code will test that installation of jax has worked:
```bash
python - << 'EOF'
import jax
print(jax.devices())
EOF
```

If you have problems creating the environment, you may need to remove it using `mamba env remove --name fultonmarket` and install it again.

We'd also like to register the kernel so it appears in JupyterLab,
```bash
python -m ipykernel install --user --name fultonmarket --display-name "Python (fultonmarket)"
```

## 4) Download and patch the Fulton Market repository
```bash
mkdir -p ~/github
cd ~/github
git clone https://github.com/CCBatIIT/FultonMarket.git
```

Once these commands are complete, you may end the interactive GPU job with the `exit` command.

## 5) Start a JupyterLab session

We will run this notebook using the `shared` partition,
```bash
/cm/shared/apps/sdsc/galyleo/galyleo launch --account iit130 --partition shared --cpus 4 --memory 8 --time-limit 02:00:00 --interface lab --conda-env fultonmarket --conda-init "$HOME/miniconda3/etc/profile.d/conda.sh"
```
The notebook itself does not need a GPU, but will submit a batch job to run Fulton Market on a GPU.

# Part 1 - Running Fulton Market

#### --> Change the list below to the name of your system [in the class spreadsheet](https://iit0-my.sharepoint.com/:x:/r/personal/dminh_illinoistech_edu/Documents/Classes/Chem456/2026S/shared/A2AR%20ligands.xlsx?d=w1ddec8f4d384415787385c7a300de8c1&csf=1&web=1&e=j7Zz1J) and enter your email address.

In [ ]:
MM_systems = ['A1H3H']
email_address = ''

The code below will generate a job script and submit a batch job for each system using the printed `sbatch` command. Only run this once.

In [ ]:
if email_address != '':
    email_lines = f"""#SBATCH --mail-user={email_address}
#SBATCH --mail-type=ALL"""

import os, shutil
for MM_system in MM_systems:
    input_dir = os.path.expanduser(f'~/exercises/10/{MM_system}/input')
    output_dir = os.path.expanduser(f'~/exercises/10/{MM_system}')
    ! mkdir -p {input_dir}
    ! mkdir -p {output_dir}
    for (source_FN, dest_FN) in [
        (f'~/exercises/07/systems/{MM_system}.xml',f'{input_dir}/{MM_system}_sys.xml'),
        (f'~/exercises/08/{MM_system}/Step_5.pdb',f'{input_dir}/{MM_system}.pdb'),
        (f'~/exercises/08/{MM_system}/Step_5.xml',f'{input_dir}/{MM_system}_state.xml')]:
        source_FN = os.path.expanduser(source_FN)
        dest_FN = os.path.expanduser(dest_FN)
        if not os.path.isfile(source_FN):
            raise Exception(f'Source file {source_FN} not found!')
        if not os.path.isfile(dest_FN):
            shutil.copy(source_FN, dest_FN)

    with open(f"10/fm-{MM_system}.py","w") as f:
        f.write(f"""import os, sys
os.environ["JAX_PLATFORMS"] = "cpu"
from chimpss.fultonmarket import FultonMarket

market = FultonMarket(
    input_system = '{input_dir}/{MM_system}_sys.xml',
    input_pdb = '{input_dir}/{MM_system}.pdb',
    input_state = '{input_dir}/{MM_system}_state.xml',
    n_replicates=125)

# Iteration length of 0.01 ns = 10 ps
# Total simulation time is 1000 ns
market.run(iter_length=0.01,
    dt=2.0,
    sim_length=25,
    total_sim_time=1000,
    output_dir='{output_dir}')
""")
    
    with open(f"10/fm-{MM_system}.job","w") as f:
        f.write(f"""#!/bin/bash
#SBATCH --account=iit130
#SBATCH --partition=gpu-shared
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=4
#SBATCH --gpus 1
#SBATCH --mem=10G
#SBATCH --time=48:00:00
#SBATCH --job-name="fm-{MM_system}"
#SBATCH --output="fm-{MM_system}.%j.%N.out"
#SBATCH --export=ALL
{email_lines}

source "$HOME/miniconda3/etc/profile.d/conda.sh"
conda activate fultonmarket
cd ~/exercises/10/
python fm-{MM_system}.py
""")

    %cd 10
    print(f"sbatch fm-{MM_system}.job")
    ! sbatch fm-{MM_system}.job
    %cd ..

Once the job is submitted, you can shut down the JupyerLab session. The maximum duration of a job is 48 hours. This calculation will probably take more than 48 hours. Once you receive an email saying that your job is complete, you should log back on to SDSC Expanse and resubmit the job from the `~/exercises/10` directory using the same `sbatch` command.

In Part 2, we will perform some analysis of this simulation. You can start this analysis once data is saved and before the job is complete.

# Part 2 - Analysis of Fulton Market Simulations

To run the analysis below, you should start another JuypterLab session on SDSC Expanse with the conda environment `fultonmarket`.

In [ ]:
import os, sys
from chimpss.fultonmarket import FultonMarketAnalysis

MM_system = MM_systems[0]
ref_pdb_fn = os.path.expanduser(f'~/exercises/08/{MM_system}/Step_4.pdb')
fma = FultonMarketAnalysis(os.path.expanduser(f'~/exercises/10/{MM_system}'), \
                           pdb = os.path.expanduser(ref_pdb_fn))
fma.equilibration_method = 'energy'
fma.resids = None

## Replica exchange

The following shows the mean energy across all replicas over the course of the simulation. The time series is used to estimate an equilbration time.

In [ ]:
fma.get_average_energy(plot=True);

The following shows the thermodynamic state index of every fifth replica from the last completed subsimulation.

In [ ]:
plt.plot(fma.state_inds[-1][:,::5]);
plt.xlabel('Iteration')
plt.ylabel('State index');

#### --> Does the state index change with the iteration? Why?

The following shows the energy distribution within each thermodynamic state.

In [ ]:
fma.plot_energy_distributions();

#### --> Does the energy between different temperatures overlap?

The following code plots the box volumes for every 10th thermodynamic state. They should fluctuate and higher temperatures should have larger volumes.

In [ ]:
fma._load_positions_box_vecs()
box_vectors = fma._reshape_list(fma.box_vectors)
fma.map = np.array(fma.map, dtype=int)
box_volumes = np.zeros((fma.map.shape[0], fma.map.shape[1]))
for iteration in range(fma.map.shape[0]):
    for state in range(fma.map.shape[1]):
        sim_no, sim_iter, sim_rep_ind = fma.map[iteration, state]
        box_volumes[iteration, state] = np.product(np.diag(box_vectors[sim_no][sim_iter, sim_rep_ind]))

from matplotlib import colormaps
from cycler import cycler

ncolors = int(np.ceil(box_volumes.shape[1]/10.))
cmap = colormaps['cool'].resampled(ncolors)
color_cycle = [cmap(n) for n in range(ncolors)]

fig, ax = plt.subplots()
ax.set_prop_cycle(cycler(color=color_cycle))
plt.plot(box_volumes[:,::10]);
plt.xlabel('Iteration')
plt.ylabel('Volume (A$^3$)');

## Sampling importance resampling

We have learned about importance sampling, where simulations in one state are used to estimate expectation values in other states. In sampling importance resampling, samples are resamples from simulations in other states. We will use sampling importance resampling to select conformations of the complex from multiple replica exchange temperatures.

The following shows the aggregate weight at the lowest temperature of simulations at each temperature.

In [ ]:
fma.resids = None
fma.importance_resampling()
# Weights per state
fma.plot_weights()

In [ ]:
plt.plot(fma.temperatures, np.sum(fma.reshaped_weights,axis=1), '.-')
plt.ylabel('Total weight in lowest-temperature state');
plt.xlabel('Simulation temperature')

#### --> How does the sum of weights vary with temperature? Why?

By setting `ensemble` to `low`, `high`, or `resampled` below, you can view frames from the three distributions.

In [ ]:
ensemble = 'resampled' # The ensemble being sampled
nframes = 50 # The number of (evenly spaced) trajectory frames to keep

# Load trajectory and box vectors into MDAnalysis Universe
import MDAnalysis as mda
from MDAnalysis.coordinates.memory import MemoryReader

u_full = mda.Universe(ref_pdb_fn)
sel_complex_from_full = u_full.select_atoms('protein or resname UNK')
u_complex = mda.Merge(u_full.select_atoms('protein or resname UNK')) # Creates universe from selected atom group
u_complex_o = u_complex # Initial frame as reference

if ensemble=='low':
    sim_idx = fma.map[::int(fma.map.shape[0]/nframes+1),0,:]
elif ensemble=='high':
    sim_idx = fma.map[::int(fma.map.shape[0]/nframes+1),-1,:]
elif ensemble=='resampled':
    sim_idx = [fma.map[iteration, state].astype(int) \
        for (state, iteration) in fma.resampled_inds[:nframes]]
pos = np.array([fma.positions[s[0]][s[1],s[2],sel_complex_from_full.indices] for s in sim_idx])*10
u_complex.load_new(pos, format=MemoryReader)

boxes = np.array([list(np.diag(fma.box_vectors[s[0]][s[1],s[2],:,:]*10)) + [90,90,90] for s in sim_idx])
for (ts, box) in zip(u_complex.trajectory, boxes):
    ts.dimensions = box

# Apply transformations so that the protein and ligand appear together
from MDAnalysis.analysis import align
from MDAnalysis import transformations
from MDAnalysis.transformations.translate import center_in_box

transform_1 = center_in_box(u_complex.select_atoms("protein")) # Put the protein in the center
transform_2 = transformations.wrap(u_complex.select_atoms("resname UNK")) # Put other molecules in the box
transform_3 = transformations.unwrap(u_complex.select_atoms("resname UNK")) # Prevent molecules from spanning the box
u_complex.trajectory.add_transformations(*[transform_1, transform_2, transform_3])

# Show the trajectory
import nglview as nv
view = nv.show_mdanalysis(u_complex)
view.display(gui=True)

#### --> Describe differences between the low-temperature, high-temperature, and resampled (at low temperature) ensembles?